<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module34a/Lab03.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 3 — R$_{XX}$, R$_{YY}$, and the Configuration Mixer

**Maps to:** Module 3, Lessons 3–4 (Two-Qubit Rotation Gates; Preparing the Mix)

**Time:** ~60 minutes (instructor walkthrough ~15 min)

---

### The question this lab answers

We want a circuit that moves amplitude between $|10\rangle$ and $|01\rangle$ — an
electron hopping between the bonding and antibonding configuration — **without**
touching $|00\rangle$ and $|11\rangle$. Lab 2 showed R$_{ZZ}$ cannot do it (pure phase)
and that $XX+YY$ is not a gate. This lab builds the answer:

$$ \mathrm{R}_{XX}(\theta)\,\mathrm{R}_{YY}(\theta)
\;=\; e^{-i\frac{\theta}{2}X_0X_1}\, e^{-i\frac{\theta}{2}Y_0Y_1}
\;=\; e^{-i\frac{\theta}{2}(X_0X_1 + Y_0Y_1)} .$$

(The last equality holds because $X_0X_1$ and $Y_0Y_1$ **commute** — you will check that.)

### After this lab you can
1. Verify the basis-change identities $HZH = X$ and $SHZHS^\dagger = Y$.
2. Build R$_{XX}$ and R$_{YY}$ by sandwiching the R$_{ZZ}$ circuit.
3. Show the product mixes $|10\rangle \leftrightarrow |01\rangle$ and leaves
   $|00\rangle, |11\rangle$ **exactly** unchanged.
4. Explain *why* the two effects cancel in one sector and reinforce in the other.
5. Spot the leftover $-i$ phase — the detail that decides which ansatz Module 4 uses.

In [ ]:
# %pip install -q qiskit qiskit-aer matplotlib
import numpy as np
import scipy.linalg as la
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector

np.set_printoptions(precision=3, suppress=True)

I2 = np.eye(2, dtype=complex)
X  = np.array([[0, 1], [1, 0]], dtype=complex)
Y  = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z  = np.array([[1, 0], [0, -1]], dtype=complex)
H  = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
S  = np.array([[1, 0], [0, 1j]], dtype=complex)
Sdg = S.conj().T

XX, YY, ZZ = np.kron(X, X), np.kron(Y, Y), np.kron(Z, Z)

basis = {"|00>": np.array([1,0,0,0], dtype=complex),
         "|01>": np.array([0,1,0,0], dtype=complex),
         "|10>": np.array([0,0,1,0], dtype=complex),
         "|11>": np.array([0,0,0,1], dtype=complex)}

def is_unitary(U, tol=1e-10):
    U = np.asarray(U, dtype=complex)
    return np.allclose(U.conj().T @ U, np.eye(U.shape[0]), atol=tol)

print("setup ok")

## 1. Basis changes: $H$ turns $Z$ into $X$

The Hadamard swaps the roles of the $x$ and $z$ axes on the Bloch sphere:

$$ H Z H = X, \qquad S H Z H S^\dagger = S X S^\dagger = Y .$$

This is the single most reusable trick in the course. It says: **anything you can do
along $Z$, you can do along $X$ or $Y$ by wrapping it in cheap one-qubit gates.**
You will use it again in Lab 4 to *measure* $\langle XX\rangle$ and $\langle YY\rangle$
on hardware that can only measure $Z$.

### Exercise 1 — verify both identities

In [ ]:
print("H Z H      == X ?", ...)          # TODO
print("S X S^dag  == Y ?", ...)          # TODO
print("S H Z H S^dag == Y ?", ...)       # TODO
assert np.allclose(H @ Z @ H, X) and np.allclose(S @ H @ Z @ H @ Sdg, Y)
print("PASS")

## 2. Do $X_0X_1$ and $Y_0Y_1$ commute?

If two Hermitian operators $A$ and $B$ commute ($AB = BA$), then
$e^{A}e^{B} = e^{A+B}$ — the same rule as for ordinary numbers. If they do not, the
product of exponentials is *not* the exponential of the sum, and everything below would
fall apart.

In [ ]:
comm = XX @ YY - YY @ XX
print("[XX, YY] =\n", comm.real)
print("\ncommute?", np.allclose(comm, 0))

lhs = la.expm(-1j*0.6/2 * XX) @ la.expm(-1j*0.6/2 * YY)
rhs = la.expm(-1j*0.6/2 * (XX + YY))
print("exp(A)exp(B) == exp(A+B) ?", np.allclose(lhs, rhs))
assert np.allclose(comm, 0) and np.allclose(lhs, rhs)
print("\nPASS -- so RXX(theta) RYY(theta) really is exp(-i theta/2 (XX+YY)),")
print("which is the unitary version of the 'XX+YY' idea from the lecture.")

## 3. Build R$_{XX}$ and R$_{YY}$ as circuits

Wrap the Lab 2 circuit in basis changes. In Qiskit, gates are applied **left to right**
in the order you write them:

| gate | circuit |
|---|---|
| R$_{ZZ}(\theta)$ | `cx(0,1) · rz(θ,1) · cx(0,1)` |
| R$_{XX}(\theta)$ | `H` on both → R$_{ZZ}(\theta)$ → `H` on both |
| R$_{YY}(\theta)$ | `S†,H` on both → R$_{ZZ}(\theta)$ → `H,S` on both |

### Exercise 2 — implement all three and check against `expm`

In [ ]:
def rzz_circuit(qc, theta, a=0, b=1):
    qc.cx(a, b); qc.rz(theta, b); qc.cx(a, b)
    return qc

def rxx_circuit(theta):
    qc = QuantumCircuit(2, name="RXX")
    # TODO: H on both qubits, then rzz_circuit(qc, theta), then H on both again
    ...
    return qc

def ryy_circuit(theta):
    qc = QuantumCircuit(2, name="RYY")
    # TODO: sdg then h on both, rzz_circuit(qc, theta), then h then s on both
    ...
    return qc

theta = 0.6
print(rxx_circuit(theta).draw(output="text"))
print(ryy_circuit(theta).draw(output="text"))

assert np.allclose(Operator(rxx_circuit(theta)).data, la.expm(-1j*theta/2 * XX))
assert np.allclose(Operator(ryy_circuit(theta)).data, la.expm(-1j*theta/2 * YY))
print("\nPASS: both circuits match exp(-i theta/2 P) exactly.")

## 4. What each one does on its own

### Exercise 3 — apply R$_{XX}$ alone, then R$_{YY}$ alone, to each basis state

Print the output amplitudes. Watch the $|00\rangle/|11\rangle$ sector.

In [ ]:
def show(U, title):
    print(title)
    for name, b in basis.items():
        print(f"   {name} -> {np.round(U @ b, 3)}")
    print()

theta = 0.6
show(la.expm(-1j*theta/2 * XX), f"RXX({theta}):")
show(la.expm(-1j*theta/2 * YY), f"RYY({theta}):")

Look at $|00\rangle$:

* R$_{XX}$ sends it to $\cos\frac{\theta}{2}|00\rangle - i\sin\frac{\theta}{2}|11\rangle$.
* R$_{YY}$ sends it to $\cos\frac{\theta}{2}|00\rangle \color{red}{+} i\sin\frac{\theta}{2}|11\rangle$.

**The leaked amplitudes have opposite signs.** Apply both and they cancel. In the
$|01\rangle/|10\rangle$ sector both gates leak with the *same* sign, so applying both
**doubles** the effect. That is the whole trick on the summary slide.

## 5. The mixer

### Exercise 4 — the payoff

In [ ]:
def mixer_circuit(theta):
    '''RXX(theta) followed by RYY(theta).'''
    qc = QuantumCircuit(2, name="mixer")
    # TODO: compose rxx_circuit(theta) then ryy_circuit(theta) into qc
    #       hint: qc.compose(other, inplace=True)
    ...
    return qc

theta = 0.6
U = Operator(mixer_circuit(theta)).data
print("RXX(theta) RYY(theta) =\n", np.round(U, 3))
print()
show(U, "action on the basis:")

assert np.allclose(U @ basis["|00>"], basis["|00>"]), "|00> moved!"
assert np.allclose(U @ basis["|11>"], basis["|11>"]), "|11> moved!"
p01 = np.abs(U @ basis["|01>"])**2
assert np.isclose(p01[1], np.cos(theta)**2) and np.isclose(p01[2], np.sin(theta)**2)
print("PASS")

### Exercise 5 — sweep the knob

Plot $P(|01\rangle)$ and $P(|10\rangle)$ versus $\theta$, starting from $|01\rangle$.
This is the picture of "$\theta$ determines how it mixes."

In [ ]:
import matplotlib.pyplot as plt

thetas = np.linspace(0, np.pi, 200)
p01, p10 = [], []
for t in thetas:
    out = la.expm(-1j*t/2 * (XX + YY)) @ basis["|01>"]
    p = np.abs(out)**2
    p01.append(p[1]); p10.append(p[2])

plt.figure(figsize=(6, 3.4))
plt.plot(thetas, p01, label=r"$P(|01\rangle)$")
plt.plot(thetas, p10, label=r"$P(|10\rangle)$")
plt.axvline(0.112, color="k", ls=":", lw=1)
plt.text(0.13, 0.55, r"$\theta^*\approx0.112$" + "\n(H$_2$ ground state,\n Lab 5)", fontsize=8)
plt.xlabel(r"$\theta$"); plt.ylabel("probability"); plt.legend(); plt.tight_layout(); plt.show()

print("Total probability in the {|01>,|10>} sector, for every theta:",
      np.round(np.array(p01) + np.array(p10), 6)[:5], "...")

## 6. The fine print: a leftover $-i$

Look again at the amplitude the mixer puts on $|10\rangle$: it is
$-i\sin\theta$, not $+\sin\theta$. The **probability** is right, but the amplitude sits
90° out of phase.

For a molecular Hamiltonian whose matrix elements are all real, that phase is exactly the
wrong one: the cross term that should *lower* the energy averages to zero instead.
Module 4 therefore uses a close cousin — the *Givens rotation*

$$U(\theta) = e^{-i\frac{\theta}{2}\,(X_0Y_1 - Y_0X_1)}
\quad\Longrightarrow\quad |01\rangle \mapsto \cos\theta\,|01\rangle + \sin\theta\,|10\rangle$$

— same mixing, **real** amplitudes. It is built from the same CNOT–R$_z$–CNOT core with
slightly different `S`/`H` wrappers, and it is the circuit on your "Reduced Qubit
Mapping" slide. Let us confirm the difference now so it is not a surprise in Lab 5.

In [ ]:
XY = np.kron(X, Y); YX = np.kron(Y, X)
theta = 0.6

mix   = la.expm(-1j*theta/2 * (XX + YY)) @ basis["|01>"]
given = la.expm(-1j*theta/2 * (XY - YX)) @ basis["|01>"]

print("RXX*RYY  on |01> :", np.round(mix, 3), "   <- amplitude on |10> is imaginary")
print("Givens   on |01> :", np.round(given, 3), "   <- amplitude on |10> is real")
print("\nsame probabilities?",
      np.allclose(np.abs(mix)**2, np.abs(given)**2))
print("cos(theta), sin(theta) =", np.round([np.cos(theta), np.sin(theta)], 3))

## 7. Checkpoint

1. Why must $X_0X_1$ and $Y_0Y_1$ commute for
   R$_{XX}$R$_{YY} = e^{-i\frac{\theta}{2}(XX+YY)}$ to hold?
2. In one sentence: why do the $|00\rangle \leftrightarrow |11\rangle$ leaks cancel while
   the $|01\rangle \leftrightarrow |10\rangle$ ones reinforce?
3. Count the CNOTs in `mixer_circuit`. On hardware with a 1% two-qubit error rate, roughly
   what fidelity would you expect from this block alone?
4. Set $\theta = \pi/2$. What state does the mixer produce from $|01\rangle$? Is it
   entangled?
5. Both R$_{XX}$R$_{YY}$ and the Givens rotation give the same probabilities. Name one
   quantity that would come out different, and say why it matters for energy.

### What is next
**Lab 4** stops building states and starts *measuring* them: how to get
$\langle Z_0\rangle$, $\langle X_0X_1\rangle$, $\langle Y_0Y_1\rangle$ out of a machine
that only knows how to measure $Z$ — and how many shots that costs.